#  UFC Fighters — Análisis Exploratorio y Visualizaciones Interactivas

**Asignatura:** Programación para la Ciencia de Datos — Evaluación Parcial 3
**Dataset:** UFC Fighters Stats (4.496 peleadores, estadísticas físicas y de combate)
**Pregunta de análisis:** ¿Qué características físicas y de estilo distinguen a los peleadores de UFC más exitosos?

**Visualizaciones interactivas (Plotly):** Gráfico de barras · Gráfico de líneas · Gráfico de dispersión

---

## 1.  Importar Librerías

In [35]:
import importlib
import os
import sys

# Verificar si el kernel activo ya tiene las librerías necesarias
required_packages = ['pandas', 'numpy', 'plotly', 'nbformat']
missing_packages = []

for package in required_packages:
    try:
        importlib.import_module(package)
    except Exception:
        missing_packages.append(package)

if missing_packages:
    print(f'Paquetes faltantes: {missing_packages}')
    %pip install -q pandas numpy plotly nbformat --break-system-packages --no-warn-script-location

# Agregar rutas de paquetes del usuario solo si el kernel las necesita
for path in [
    '/home/mario-escobar/.local/lib/python3.14/site-packages',
    '/home/mario-escobar/.local/lib/python3/site-packages',
    '/usr/local/lib/python3.14/dist-packages',
    '/usr/lib/python3/dist-packages'
]:
    if os.path.exists(path) and path not in sys.path:
        sys.path.insert(0, path)

# ── Manipulación de datos ──────────────────────────────────────────────────────
import pandas as pd
import numpy as np

# ── Visualización interactiva (Plotly) ─────────────────────────────────────────
import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio

# Evitar problemas de renderizado en este kernel
pio.renderers.render_on_display = False

# Configuración para que las tablas se vean ordenadas
pd.set_option('display.max_columns', 30)
pd.set_option('display.width', 120)

print('Librerías listas para usar en este kernel')

Librerías listas para usar en este kernel


## 2.  Cargar el Dataset

Usamos el archivo `ufc_fighters_stats_merged.csv`, que combina los datos físicos de cada peleador
(altura, alcance, postura) con sus estadísticas de combate (golpes por minuto, precisión, victorias por KO, etc.).

In [36]:
# Cargar el CSV — ajusta la ruta si es necesario
df = pd.read_csv('ufc_fighters_stats_merged.csv')

print(f' Shape: {df.shape[0]} filas × {df.shape[1]} columnas')
print(f'\n Columnas:')
print(list(df.columns))

 Shape: 4496 filas × 30 columnas

 Columnas:
['first', 'last', 'nickname', 'height_cm', 'weight', 'reach', 'stance', 'wins', 'losses', 'draws', 'sig_strikes_landed', 'sig_strikes_attempted', 'takedowns_landed', 'takedowns_attempted', 'str_landed_per_min', 'str_absorbed_per_min', 'td_avg_per_15min', 'sub_avg_per_15min', 'str_defense_pct', 'td_defense_pct', 'knockdown_avg', 'striking_accuracy_pct', 'takedown_accuracy_pct', 'ko_wins', 'ko_win_percentage', 'dec_wins', 'dec_win_percentage', 'sub_wins', 'sub_win_percentage', 'avg_fight_time_minutes']


In [37]:
# Vista rápida de las primeras filas
df.head()

,first,last,nickname,height_cm,weight,reach,stance,wins,losses,draws,sig_strikes_landed,sig_strikes_attempted,takedowns_landed,takedowns_attempted,str_landed_per_min,str_absorbed_per_min,td_avg_per_15min,sub_avg_per_15min,str_defense_pct,td_defense_pct,knockdown_avg,striking_accuracy_pct,takedown_accuracy_pct,ko_wins,ko_win_percentage,dec_wins,dec_win_percentage,sub_wins,sub_win_percentage,avg_fight_time_minutes
0,Tom,Aaron,NaN,NaN,155.0,NaN,NaN,5,3,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,Danny,Abbadi,The Assassin,180.34,155.0,NaN,Orthodox,4,6,0,59.0,155.0,NaN,NaN,3.29,4.41,0.0,0.0,NaN,NaN,0.0,38.0,3.29,0.0,0.0,0.0,0.0,0.0,0.0,8.966667
2,Nariman,Abbasov,Bayraktar,172.72,155.0,66.0,Orthodox,28,4,0,45.0,225.0,NaN,2.0,3.00,5.67,0.0,0.0,NaN,NaN,0.0,20.0,0.00,0.0,0.0,0.0,0.0,0.0,0.0,15.000000
3,Darion,Abbey,NaN,187.96,265.0,80.0,Orthodox,9,5,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,David,Abbott,Tank,182.88,265.0,NaN,Switch,10,15,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [38]:
# Estadísticas descriptivas de columnas numéricas
df.describe().round(2)

,height_cm,weight,reach,wins,losses,draws,sig_strikes_landed,sig_strikes_attempted,takedowns_landed,takedowns_attempted,str_landed_per_min,str_absorbed_per_min,td_avg_per_15min,sub_avg_per_15min,str_defense_pct,td_defense_pct,knockdown_avg,striking_accuracy_pct,takedown_accuracy_pct,ko_wins,ko_win_percentage,dec_wins,dec_win_percentage,sub_wins,sub_win_percentage,avg_fight_time_minutes
count,4175.00,4410.00,2553.00,4496.00,4496.00,4496.00,3011.00,3011.00,916.00,2657.00,3026.00,3026.00,3026.00,3026.00,20.00,555.00,2463.00,3026.00,3023.00,3481.00,3481.00,3481.00,3481.00,3481.00,3481.00,3014.00
mean,178.10,169.85,71.50,12.46,5.69,0.25,234.67,523.69,7.16,20.84,3.06,3.84,1.48,0.66,19.60,0.90,0.34,43.78,32.68,2.87,20.69,2.16,14.83,2.05,14.57,9.75
std,9.02,39.55,4.29,9.19,5.01,0.79,312.96,690.55,10.27,28.17,1.74,2.59,1.80,1.38,25.17,6.79,0.75,12.81,26.19,4.09,26.37,3.31,19.85,3.40,21.40,3.88
min,139.70,105.00,58.00,0.00,0.00,0.00,0.00,1.00,1.00,1.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.13
25%,172.72,145.00,69.00,7.00,2.00,0.00,35.00,82.00,1.00,4.00,1.87,2.53,0.00,0.00,0.00,0.00,0.00,38.00,7.22,0.00,0.00,0.00,0.00,0.00,0.00,7.27
50%,177.80,155.00,72.00,11.00,4.00,0.00,115.00,257.00,3.00,11.00,2.91,3.40,1.00,0.22,0.00,0.00,0.11,44.00,33.00,0.00,0.00,0.00,0.00,0.00,0.00,10.07
75%,185.42,185.00,75.00,17.00,8.00,0.00,311.00,696.50,8.00,26.00,3.98,4.57,2.15,0.84,51.00,0.00,0.46,51.00,49.00,5.00,40.00,4.00,28.00,3.00,26.00,12.60
max,226.06,770.00,84.00,253.00,83.00,11.00,3681.00,7647.00,90.00,338.00,17.65,52.50,32.14,21.95,57.00,79.00,20.27,100.00,100.00,28.00,100.00,32.00,100.00,47.00,100.00,25.00


## 3.  Exploración de Calidad de los Datos

Antes de limpiar, revisamos **cuántos nulos** tiene cada columna. Esto define qué columnas podemos usar
y cuáles hay que descartar.

In [39]:
# ── Porcentaje de valores nulos por columna ────────────────────────────────────
nulos = (df.isnull().mean() * 100).round(1).sort_values(ascending=False)
print('Porcentaje de nulos por columna:\n')
print(nulos.to_string())

Porcentaje de nulos por columna:

str_defense_pct           99.6
td_defense_pct            87.7
takedowns_landed          79.6
knockdown_avg             45.2
nickname                  44.6
reach                     43.2
takedowns_attempted       40.9
avg_fight_time_minutes    33.0
sig_strikes_attempted     33.0
sig_strikes_landed        33.0
takedown_accuracy_pct     32.8
str_landed_per_min        32.7
td_avg_per_15min          32.7
sub_avg_per_15min         32.7
striking_accuracy_pct     32.7
str_absorbed_per_min      32.7
ko_win_percentage         22.6
ko_wins                   22.6
dec_wins                  22.6
dec_win_percentage        22.6
sub_wins                  22.6
sub_win_percentage        22.6
stance                    18.9
height_cm                  7.1
weight                     1.9
first                      0.4
losses                     0.0
wins                       0.0
draws                      0.0
last                       0.0


**Observación importante:**

- Tres columnas están casi vacías: `str_defense_pct` (99.6% nulos), `td_defense_pct` (87.7%) y `takedowns_landed` (79.6%).
  **No se pueden usar** — imputarlas significaría inventar el 80-99% de los datos, lo cual no es válido.
- Las estadísticas de golpeo (`str_landed_per_min`, `striking_accuracy_pct`, etc.) tienen ~33% de nulos.
  Esto es manejable: son peleadores antiguos sin registro detallado de golpes. Los trataremos según cada análisis.
- Los datos básicos (`wins`, `losses`, `height_cm`, `weight`) están casi completos.

## 4.  Limpieza y Preparación de los Datos

Aplicamos cuatro pasos de limpieza, justificando cada decisión.

In [40]:
# ── Paso 1: Descartar columnas con demasiados nulos (>85%) ─────────────────────
# No se pueden imputar de forma honesta, así que las eliminamos
columnas_inservibles = ['str_defense_pct', 'td_defense_pct', 'takedowns_landed']
df = df.drop(columns=columnas_inservibles)
print(f'Columnas descartadas por exceso de nulos: {columnas_inservibles}')

# ── Paso 2: Crear columna de nombre completo ───────────────────────────────────
df['name'] = (df['first'].fillna('') + ' ' + df['last'].fillna('')).str.strip()

# ── Paso 3: Crear variables derivadas útiles ───────────────────────────────────
# Total de peleas y tasa de victorias (win rate)
df['total_fights'] = df['wins'] + df['losses'] + df['draws']

# Evitamos dividir por cero: solo calculamos win_rate si hay al menos 1 pelea
df['win_rate'] = np.where(df['total_fights'] > 0,
                          df['wins'] / df['total_fights'],
                          np.nan)

print('Variables creadas: name, total_fights, win_rate')

Columnas descartadas por exceso de nulos: ['str_defense_pct', 'td_defense_pct', 'takedowns_landed']
Variables creadas: name, total_fights, win_rate


In [41]:
# ── Paso 4: Agrupar posturas (stance) poco frecuentes ──────────────────────────
# 'Open Stance' (7) y 'Sideways' (3) son muy raras → las agrupamos en 'Other'
df['stance'] = df['stance'].replace({'Open Stance': 'Other', 'Sideways': 'Other'})

print('Distribución de postura (stance) tras limpieza:')
print(df['stance'].value_counts(dropna=False))

# ── Tratar el peso físicamente implausible (770 lb ≈ 349 kg) ───────────────────
# El valor de 770 lb es sospechoso (probable error de captura). No lo borramos del
# dataset original, pero creamos un subconjunto acotado a un rango realista (≤ 300 lb
# cubre todas las divisiones oficiales de UFC) para los análisis de peso.
df_peso = df[df['weight'] <= 300].copy()
print(f'Peleadores en rango de peso realista (≤300 lb): {len(df_peso)} de {len(df)}')
print(f'Excluidos por peso atípico: {len(df) - len(df_peso)}')

Distribución de postura (stance) tras limpieza:
stance
Orthodox    2797
NaN          851
Southpaw     616
Switch       222
Other         10
Name: count, dtype: int64
Peleadores en rango de peso realista (≤300 lb): 4389 de 4496
Excluidos por peso atípico: 107


**Sobre los pesos muy altos:** El dataset registra pesos en libras de hasta 770 lb (~349 kg), un valor físicamente implausible y probablemente un error de captura. No alteramos los datos originales, pero para los análisis de peso usamos un subconjunto acotado a ≤ 300 lb (rango que cubre todas las divisiones oficiales de UFC), evitando que ese valor atípico distorsione los gráficos.

In [42]:
# ── Verificación final de la limpieza ──────────────────────────────────────────
print(f'Dataset limpio: {df.shape[0]} filas × {df.shape[1]} columnas')
print(f'Peleadores con al menos 1 pelea registrada: {(df["total_fights"] > 0).sum()}')
print(f'Peleadores con stats de golpeo completas: {df["str_landed_per_min"].notna().sum()}')

Dataset limpio: 4496 filas × 30 columnas
Peleadores con al menos 1 pelea registrada: 4477
Peleadores con stats de golpeo completas: 3026


---
## 5.  Mini Dashboard — Resultados Principales

A continuación presentamos las **tres visualizaciones interactivas** principales construidas con **Plotly**.
Cada una responde a una parte de nuestra pregunta y se acompaña de su interpretación.

| # | Tipo de gráfico | Qué responde |
|---|---|---|
| 1 | Barras | ¿Qué postura (stance) tiene mejor tasa de victorias? |
| 2 | Líneas | ¿Cómo se relaciona el alcance con el peso del peleador? |
| 3 | Dispersión | ¿Volumen vs. precisión de golpeo distingue a los peleadores? |
| 4 | Barras | ¿Cómo ganan los peleadores: KO, sumisión o decisión? |
| 5 | Dispersión | ¿Golpear más implica absorber más castigo? |

### 5.1  Gráfico de Barras — Tasa de Victorias por Postura

Comparamos la **tasa de victorias promedio** según la postura del peleador.
Filtramos a peleadores con **al menos 5 peleas** para que el promedio sea confiable
(un peleador con 1 pelea ganada tendría 100% de win rate, lo cual no es representativo).

In [57]:
# ── Preparar datos: win rate medio por stance (peleadores con >=5 peleas) ──────
exp = df[(df['total_fights'] >= 5) & (df['stance'].notna())].copy()

win_por_stance = (exp.groupby('stance')['win_rate']
                     .mean()
                     .reset_index()
                     .sort_values('win_rate', ascending=False))
win_por_stance['win_rate_pct'] = (win_por_stance['win_rate'] * 100).round(1)

# ── Gráfico de barras interactivo con Plotly Express ───────────────────────────
fig1 = px.bar(
    win_por_stance,
    x='stance',
    y='win_rate_pct',
    color='stance',
    text='win_rate_pct',
    title='Tasa de Victorias Promedio según Postura (peleadores con ≥5 peleas)',
    labels={'stance': 'Postura', 'win_rate_pct': 'Tasa de victorias (%)'},
    color_discrete_sequence=px.colors.qualitative.Set2
)
fig1.update_traces(texttemplate='%{text}%', textposition='outside')
fig1.update_layout(showlegend=False, yaxis_range=[0, 85])
fig1.show()  # Mostrar automáticamente

** Interpretación:**
La postura **Switch** (peleadores que cambian de guardia) muestra la tasa de victorias más alta (~74%),
seguida de cerca por **Southpaw** y **Orthodox** (~70%). La diferencia es moderada pero consistente:
la versatilidad de cambiar de postura parece dar una pequeña ventaja, posiblemente porque vuelve al peleador
más impredecible. Sin embargo, la mayoría de peleadores son Orthodox, así que la muestra de Switch es menor.

### 5.2  Gráfico de Líneas — Alcance Promedio según Peso

Analizamos cómo cambia el **alcance promedio** (reach) a medida que aumenta el **peso** del peleador.
Agrupamos el peso en rangos (divisiones aproximadas) para ver la tendencia con claridad.

In [58]:
# ── Preparar datos: alcance medio por rango de peso ────────────────────────────
# Usamos df_peso (rango realista <=300 lb) y filtramos a quienes tienen reach registrado
peso_reach = df_peso[df_peso['reach'].notna()].copy()

# Crear rangos de peso de 20 en 20 libras
peso_reach['rango_peso'] = pd.cut(peso_reach['weight'],
                                  bins=range(100, 320, 20))
linea = (peso_reach.groupby('rango_peso', observed=True)['reach']
                   .mean()
                   .reset_index())
# Usar el punto medio del rango como eje X numérico
linea['peso_medio'] = linea['rango_peso'].apply(lambda x: x.mid).astype(float)

# ── Gráfico de líneas interactivo ──────────────────────────────────────────────
fig2 = px.line(
    linea,
    x='peso_medio',
    y='reach',
    markers=True,
    title='Alcance Promedio (reach) según el Peso del Peleador',
    labels={'peso_medio': 'Peso (libras)', 'reach': 'Alcance promedio (pulgadas)'}
)
fig2.update_traces(line=dict(width=3), marker=dict(size=9))
fig2.show() # Mostrar automáticamente

** Interpretación:**
Existe una relación **creciente y casi lineal**: a mayor peso, mayor alcance promedio. Esto es esperable
físicamente (peleadores más pesados tienden a ser más altos y con brazos más largos), pero el gráfico lo
**cuantifica**: el alcance pasa de ~68 pulgadas en pesos ligeros a más de 75 pulgadas en pesos pesados.
La tendencia se mantiene estable, lo que confirma que el peso es un buen predictor del alcance.

### 5.3  Gráfico de Dispersión — Volumen vs. Precisión de Golpeo

Este es el gráfico central. Cruzamos dos métricas de golpeo:
- **Eje X:** golpes significativos conectados por minuto (volumen de ataque)
- **Eje Y:** precisión de golpeo (% de golpes que aciertan)

Coloreamos por postura y el tamaño representa el total de peleas, para ver si hay un perfil dominante.

In [54]:
# ── Preparar datos: solo peleadores con stats de golpeo y algo de experiencia ──
scatter_df = df[
    df['str_landed_per_min'].notna() &
    df['striking_accuracy_pct'].notna() &
    df['stance'].notna() &
    (df['total_fights'] >= 3)
].copy()

# Filtrar valores extremos poco representativos para que el gráfico sea legible
scatter_df = scatter_df[
    (scatter_df['str_landed_per_min'] <= 12) &
    (scatter_df['striking_accuracy_pct'] <= 100)
]

# ── Muestra aleatoria para no saturar el gráfico ───────────────────────────────
scatter_df = scatter_df.sample(n=1000, random_state=42)

# ── Gráfico de dispersión interactivo (estilo del ejemplo del curso) ───────────
fig3 = px.scatter(
    scatter_df,
    x='str_landed_per_min',
    y='striking_accuracy_pct',
    color='stance',
    size='total_fights',
    hover_name='name',
    hover_data={'wins': True, 'losses': True, 'total_fights': True},
    title='Volumen vs. Precisión de Golpeo por Postura (muestra de 100 peleadores)',
    labels={
        'str_landed_per_min': 'Golpes conectados por minuto',
        'striking_accuracy_pct': 'Precisión de golpeo (%)',
        'stance': 'Postura'
    },
    opacity=0.6,
    color_discrete_sequence=px.colors.qualitative.Bold,
    render_mode='svg'
)
fig3.show() # Mostrar automáticamente

** Interpretación:**
La nube de puntos muestra que **no hay correlación fuerte** entre volumen y precisión: hay peleadores que
golpean mucho con baja precisión y otros que golpean poco pero certero. La mayoría se concentra entre
**2 y 5 golpes/min** con **40-55% de precisión**. Las posturas se mezclan sin formar grupos separados, lo
que sugiere que el estilo de golpeo depende más del peleador individual que de su postura. Pasa el cursor
sobre los puntos grandes (más peleas) para identificar a los veteranos del deporte.

### 5.4  Gráfico de Barras — ¿Cómo Ganan los Peleadores?

Más allá de cuánto ganan, nos interesa **cómo** ganan. Comparamos el porcentaje promedio de victorias
logradas por **KO/TKO** (nocaut), **sumisión** (rendición) y **decisión** (por puntos de los jueces).

In [46]:
# ── Promedio de cada tipo de victoria (peleadores con desglose registrado) ─────
tipos = df.dropna(subset=['ko_win_percentage', 'sub_win_percentage', 'dec_win_percentage'])

medias = pd.DataFrame({
    'Tipo de victoria': ['KO / TKO', 'Sumisión', 'Decisión'],
    'Porcentaje promedio': [
        tipos['ko_win_percentage'].mean(),
        tipos['sub_win_percentage'].mean(),
        tipos['dec_win_percentage'].mean()
    ]
})
medias['Porcentaje promedio'] = medias['Porcentaje promedio'].round(1)

# ── Gráfico de barras interactivo ──────────────────────────────────────────────
fig4 = px.bar(
    medias,
    x='Tipo de victoria',
    y='Porcentaje promedio',
    color='Tipo de victoria',
    text='Porcentaje promedio',
    title='Cómo Ganan los Peleadores de UFC (promedio por tipo de victoria)',
    labels={'Porcentaje promedio': 'Porcentaje promedio de victorias (%)'},
    color_discrete_sequence=px.colors.qualitative.Pastel
)
fig4.update_traces(texttemplate='%{text}%', textposition='outside')
fig4.update_layout(showlegend=False)
fig4.show()

 **Interpretación:**
El **KO/TKO es la forma de victoria más frecuente** (~21% promedio), por encima de sumisión (~15%) y
decisión (~15%). Esto refleja que el golpeo de poder es el camino más común a la victoria en UFC.
Que sumisión y decisión estén casi empatadas sugiere dos perfiles distintos pero igual de presentes:
los especialistas en suelo que buscan la rendición, y los peleadores que llevan el combate a los puntos.

### 5.5  Gráfico de Dispersión — Golpes Conectados vs. Absorbidos

¿Los peleadores que más golpean son también los que más castigo reciben? Cruzamos los golpes que un
peleador **conecta** por minuto contra los que **absorbe** por minuto, para ver si existe un perfil
"agresivo pero vulnerable" o si hay peleadores que golpean mucho sin recibir.

In [55]:
# ── Preparar datos: filtrar valores extremos para legibilidad ─────────────────
golpeo = df.dropna(subset=['str_landed_per_min', 'str_absorbed_per_min', 'stance']).copy()
golpeo = golpeo[
    (golpeo['str_landed_per_min'] <= 12) &
    (golpeo['str_absorbed_per_min'] <= 12) &
    (golpeo['total_fights'] >= 3)
]

# ── Muestra aleatoria para no saturar el gráfico ───────────────────────────────
golpeo = golpeo.sample(n=1000, random_state=42)

# ── Gráfico de dispersión interactivo ──────────────────────────────────────────
fig5 = px.scatter(
    golpeo,
    x='str_landed_per_min',
    y='str_absorbed_per_min',
    color='stance',
    hover_name='name',
    hover_data={'wins': True, 'losses': True},
    title='Golpes Conectados vs. Absorbidos por Minuto (muestra de 100 peleadores)',
    labels={
        'str_landed_per_min': 'Golpes conectados por minuto',
        'str_absorbed_per_min': 'Golpes absorbidos por minuto',
        'stance': 'Postura'
    },
    opacity=0.5,
    color_discrete_sequence=px.colors.qualitative.Bold,
    render_mode='svg'
)
fig5.add_shape(type='line', x0=0, y0=0, x1=12, y1=12,
               line=dict(dash='dash', color='gray'))
fig5.show()

In [48]:
from IPython.display import HTML

# ── Métricas para las tarjetas KPI ─────────────────────────────────────────────
total_fighters = len(df)
avg_win_rate = (df['win_rate'].mean() * 100)
most_common_stance = df['stance'].mode()[0]
avg_height = df['height_cm'].mean()

# ── HTML/CSS para tarjetas KPI tipo dashboard ──────────────────────────────────
kpi_html = f"""
<style>
  .kpi-container {{
    display: grid;
    grid-template-columns: repeat(4, 1fr);
    gap: 20px;
    margin: 20px 0;
    font-family: -apple-system, BlinkMacSystemFont, 'Segoe UI', Arial, sans-serif;
  }}
  
  .kpi-card {{
    background: linear-gradient(135deg, #667eea 0%, #764ba2 100%);
    color: white;
    padding: 24px;
    border-radius: 12px;
    text-align: center;
    box-shadow: 0 4px 15px rgba(0, 0, 0, 0.15);
    transition: transform 0.3s ease;
  }}
  
  .kpi-card:hover {{
    transform: translateY(-5px);
    box-shadow: 0 8px 25px rgba(0, 0, 0, 0.2);
  }}
  
  .kpi-card:nth-child(1) {{
    background: linear-gradient(135deg, #667eea 0%, #764ba2 100%);
  }}
  
  .kpi-card:nth-child(2) {{
    background: linear-gradient(135deg, #f093fb 0%, #f5576c 100%);
  }}
  
  .kpi-card:nth-child(3) {{
    background: linear-gradient(135deg, #4facfe 0%, #00f2fe 100%);
  }}
  
  .kpi-card:nth-child(4) {{
    background: linear-gradient(135deg, #43e97b 0%, #38f9d7 100%);
  }}
  
  .kpi-value {{
    font-size: 32px;
    font-weight: bold;
    margin: 12px 0;
    font-family: 'Courier New', monospace;
  }}
  
  .kpi-label {{
    font-size: 14px;
    opacity: 0.9;
    text-transform: uppercase;
    letter-spacing: 1px;
    font-weight: 600;
  }}
  
  .kpi-subtitle {{
    font-size: 12px;
    opacity: 0.75;
    margin-top: 8px;
  }}
  
  @media (max-width: 1200px) {{
    .kpi-container {{
      grid-template-columns: repeat(2, 1fr);
    }}
  }}
  
  @media (max-width: 768px) {{
    .kpi-container {{
      grid-template-columns: 1fr;
    }}
  }}
</style>

<div class="kpi-container">
  <div class="kpi-card">
    <div class="kpi-label">Total de Peleadores</div>
    <div class="kpi-value">{total_fighters:,}</div>
    <div class="kpi-subtitle">en el dataset</div>
  </div>
  
  <div class="kpi-card">
    <div class="kpi-label">Tasa de Victorias</div>
    <div class="kpi-value">{avg_win_rate:.1f}%</div>
    <div class="kpi-subtitle">promedio general</div>
  </div>
  
  <div class="kpi-card">
    <div class="kpi-label">Postura Más Común</div>
    <div class="kpi-value">{most_common_stance}</div>
    <div class="kpi-subtitle">en toda la UFC</div>
  </div>
  
  <div class="kpi-card">
    <div class="kpi-label">Altura Promedio</div>
    <div class="kpi-value">{avg_height:.1f}</div>
    <div class="kpi-subtitle">centímetros</div>
  </div>
</div>
"""

HTML(kpi_html)

In [56]:
from plotly.subplots import make_subplots

# ── Preparar datos para los 4 gráficos del dashboard combinado ─────────────────

# Gráfico 1: Tasa de victorias por postura (datos ya listos de fig1)
# Usamos los mismos datos que en fig1

# Gráfico 2: Alcance vs peso (datos ya listos de fig2)
# Reutilizamos los mismos datos

# Gráfico 3: Volumen vs precisión (scatter)
scatter_data = df[
    df['str_landed_per_min'].notna() &
    df['striking_accuracy_pct'].notna() &
    df['stance'].notna() &
    (df['total_fights'] >= 3)
].copy()
scatter_data = scatter_data[
    (scatter_data['str_landed_per_min'] <= 12) &
    (scatter_data['striking_accuracy_pct'] <= 100)
]

# ── Muestra aleatoria para consistencia con fig3 ────────────────────────────
scatter_data = scatter_data.sample(n=min(1000, len(scatter_data)), random_state=42)

# Gráfico 4: Tipos de victoria (datos ya listos de fig4)
tipos = df.dropna(subset=['ko_win_percentage', 'sub_win_percentage', 'dec_win_percentage'])
medias_victory = pd.DataFrame({
    'Tipo': ['KO / TKO', 'Sumisión', 'Decisión'],
    'Porcentaje': [
        tipos['ko_win_percentage'].mean(),
        tipos['sub_win_percentage'].mean(),
        tipos['dec_win_percentage'].mean()
    ]
})

# ── Crear figura con subplots 2x2 ──────────────────────────────────────────────
fig_dashboard = make_subplots(
    rows=2, cols=2,
    subplot_titles=(
        'Tasa de Victorias por Postura',
        'Alcance vs. Peso',
        'Volumen vs. Precisión de Golpeo',
        'Tipos de Victoria'
    ),
    specs=[
        [{'type': 'bar'}, {'type': 'scatter'}],
        [{'type': 'scatter'}, {'type': 'bar'}]
    ],
    vertical_spacing=0.12,
    horizontal_spacing=0.10
)

# ── Subplot 1 (arriba izq): Barras — Win Rate por Postura ────────────────────
for idx, row in win_por_stance.iterrows():
    color_map = {
        'Orthodox': '#1f77b4',
        'Southpaw': '#ff7f0e',
        'Switch': '#2ca02c',
        'Other': '#9467bd'
    }
    fig_dashboard.add_trace(
        go.Bar(
            x=[row['stance']],
            y=[row['win_rate_pct']],
            name=row['stance'],
            text=[f"{row['win_rate_pct']:.1f}%"],
            textposition='outside',
            marker=dict(color=color_map.get(row['stance'], '#7f7f7f')),
            hovertemplate='<b>%{x}</b><br>Tasa: %{y:.1f}%<extra></extra>',
            showlegend=False
        ),
        row=1, col=1
    )

# ── Subplot 2 (arriba der): Línea — Alcance vs. Peso ───────────────────────────
fig_dashboard.add_trace(
    go.Scatter(
        x=linea['peso_medio'],
        y=linea['reach'],
        mode='lines+markers',
        name='Alcance',
        line=dict(color='#ff6b6b', width=3),
        marker=dict(size=8),
        hovertemplate='Peso: %{x:.0f} lb<br>Alcance: %{y:.1f} pul<extra></extra>',
        showlegend=False
    ),
    row=1, col=2
)

# ── Subplot 3 (abajo izq): Scatter — Volumen vs. Precisión ─────────────────────
color_stance_map = {
    'Orthodox': '#1f77b4',
    'Southpaw': '#ff7f0e',
    'Switch': '#2ca02c',
    'Other': '#9467bd'
}

for stance_val in scatter_data['stance'].unique():
    stance_subset = scatter_data[scatter_data['stance'] == stance_val]
    fig_dashboard.add_trace(
        go.Scatter(
            x=stance_subset['str_landed_per_min'],
            y=stance_subset['striking_accuracy_pct'],
            mode='markers',
            name=stance_val,
            marker=dict(
                size=5,
                color=color_stance_map.get(stance_val, '#7f7f7f'),
                opacity=0.6
            ),
            hovertemplate='<b>%{hovertext}</b><br>Golpes: %{x:.1f}/min<br>Precisión: %{y:.1f}%<extra></extra>',
            hovertext=stance_subset['name'],
            showlegend=False
        ),
        row=2, col=1
    )

# ── Subplot 4 (abajo der): Barras — Tipos de Victoria ────────────────────────
colors_victory = ['#e74c3c', '#3498db', '#f39c12']
for idx, row in medias_victory.iterrows():
    fig_dashboard.add_trace(
        go.Bar(
            x=[row['Tipo']],
            y=[row['Porcentaje']],
            name=row['Tipo'],
            text=[f"{row['Porcentaje']:.1f}%"],
            textposition='outside',
            marker=dict(color=colors_victory[idx]),
            hovertemplate='<b>%{x}</b><br>Promedio: %{y:.1f}%<extra></extra>',
            showlegend=False
        ),
        row=2, col=2
    )

# ── Actualizar ejes y layout ───────────────────────────────────────────────────
fig_dashboard.update_xaxes(title_text='Postura', row=1, col=1)
fig_dashboard.update_yaxes(title_text='Tasa (%)', row=1, col=1)

fig_dashboard.update_xaxes(title_text='Peso (libras)', row=1, col=2)
fig_dashboard.update_yaxes(title_text='Alcance (pulgadas)', row=1, col=2)

fig_dashboard.update_xaxes(title_text='Golpes/minuto', row=2, col=1)
fig_dashboard.update_yaxes(title_text='Precisión (%)', row=2, col=1)

fig_dashboard.update_xaxes(title_text='Tipo de Victoria', row=2, col=2)
fig_dashboard.update_yaxes(title_text='Promedio (%)', row=2, col=2)

# ── Actualizar layout general ──────────────────────────────────────────────────
fig_dashboard.update_layout(
    title_text='<b>Dashboard Combinado — Resumen de 4 Análisis Principales</b>',
    title_font_size=18,
    height=900,
    width=1400,
    showlegend=False,
    hovermode='closest',
    plot_bgcolor='rgba(240, 240, 245, 0.5)',
    paper_bgcolor='white'
)

fig_dashboard.show()  # Mostrar automáticamente

 **Interpretación:**
**No hay una relación fuerte** entre golpear y ser golpeado (correlación ≈ 0.20): golpear mucho NO implica
recibir mucho. La línea punteada marca el equilibrio — los puntos **por debajo** son peleadores que conectan
más de lo que absorben (perfil eficiente/defensivo), y los de **arriba** reciben más castigo del que reparten.
La mayoría se concentra en la zona baja (2-5 golpes), lo que indica que el peleador promedio mantiene un
intercambio controlado. Los puntos aislados arriba son peleadores de estilo arriesgado.

---
## 6.  Resumen del Dashboard

Combinamos los hallazgos en una tabla de resultados clave para cerrar el análisis.

In [50]:
# ── Tabla resumen de hallazgos ─────────────────────────────────────────────────
resumen = pd.DataFrame({
    'Hallazgo': [
        'Postura con mejor tasa de victorias',
        'Tasa de victorias general (Orthodox)',
        'Relación peso-alcance',
        'Precisión de golpeo más común',
        'Volumen de golpeo más común'
    ],
    'Resultado': [
        'Switch (~74%)',
        '~70%',
        'Creciente (a más peso, más alcance)',
        '40% - 55%',
        '2 - 5 golpes por minuto'
    ]
})
resumen

,Hallazgo,Resultado
0,Postura con mejor tasa de victorias,Switch (~74%)
1,Tasa de victorias general (Orthodox),~70%
2,Relación peso-alcance,"Creciente (a más peso, más alcance)"
3,Precisión de golpeo más común,40% - 55%
4,Volumen de golpeo más común,2 - 5 golpes por minuto


In [51]:
# ── Métricas globales del dataset (tarjetas tipo KPI) ──────────────────────────
print('═══════════ MÉTRICAS GLOBALES DEL DATASET ═══════════')
print(f'  Total de peleadores analizados : {len(df):,}')
print(f'  Peleadores con stats completas : {df["str_landed_per_min"].notna().sum():,}')
print(f'  Altura promedio                : {df["height_cm"].mean():.1f} cm')
print(f'  Tasa de victorias promedio     : {df["win_rate"].mean()*100:.1f}%')
print(f'  Postura más común              : {df["stance"].mode()[0]}')
print('═════════════════════════════════════════════════════')

═══════════ MÉTRICAS GLOBALES DEL DATASET ═══════════
  Total de peleadores analizados : 4,496
  Peleadores con stats completas : 3,026
  Altura promedio                : 178.1 cm
  Tasa de victorias promedio     : 66.7%
  Postura más común              : Orthodox
═════════════════════════════════════════════════════


---
## 7.  Conclusiones

**Respuesta a la pregunta:** *¿Qué características distinguen a los peleadores de UFC más exitosos?*

1. **La postura importa poco, pero la versatilidad ayuda.** Los peleadores *Switch* ganan algo más seguido (~74%),
   aunque la diferencia con Orthodox/Southpaw (~70%) es moderada.

2. **El peso predice el alcance de forma clara.** La relación es creciente y estable: es un patrón físico
   sólido que se confirma con el gráfico de líneas.



3. **No existe un "estilo de golpeo ganador" único.** El gráfico de dispersión muestra que volumen y precisión
   no están correlacionados y que las posturas se mezclan. El éxito depende más del peleador individual que de
   una sola métrica.

**Sobre el trabajo de datos:** El mayor desafío fue la **calidad de los datos** — descartamos 3 columnas
inservibles por exceso de nulos y filtramos rangos extremos, todo documentado para no inventar información.
Esto demuestra que un buen análisis empieza por una limpieza honesta y justificada.

**Herramientas usadas:** pandas (manipulación y limpieza) · Plotly Express (visualizaciones interactivas) · Git/GitHub (control de versiones).

In [ ]:
# Exportar el dataframe limpio para usarlo desde Dash
if 'df' in globals():
    cleaned_df = df.copy()
    cleaned_df['total_fights'] = cleaned_df['wins'] + cleaned_df['losses'] + cleaned_df['draws']
    cleaned_df['win_rate'] = cleaned_df['wins'] / cleaned_df['total_fights'].replace(0, pd.NA)
    cleaned_df['weight_band'] = pd.cut(
        cleaned_df['weight'],
        bins=[0, 115, 125, 135, 145, 155, 170, 185, 205, 300],
        labels=['<115', '115-125', '125-135', '135-145', '145-155', '155-170', '170-185', '185-205', '>205'],
        include_lowest=True,
    )
    cleaned_df.to_csv('ufc_fighters_stats_cleaned.csv', index=False)
    print('DataFrame limpio exportado a ufc_fighters_stats_cleaned.csv')
else:
    print('No existe la variable df en este notebook para exportar')
